# Librerias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import style
from scipy import stats
from scipy.stats import norm, expon, gamma, lognorm,chisquare, poisson, chi2
from scipy.optimize import curve_fit
import inspect
from statsmodels.distributions.empirical_distribution import ECDF
import seaborn as sns
import time
import uuid

pd.options.display.float_format = '{:.2f}'.format

# Semilla
para obtener valores igual es las simulaciones

In [ ]:
semilla = 42
np.random.seed(semilla)

# Data Set
Se crea base de perdidas con valores aleatorios

In [ ]:
# Definir los rangos y generar la población base de cuantías (Millones)
lim = [0.02, 0.198, 0.657, 3.89, 30, 154, 500]

poblacion_cuantia_perdidas = np.concatenate([
    np.random.uniform(lim[0], lim[1], 50000),
    np.random.uniform(lim[1], lim[2], 25000),
    np.random.uniform(lim[2], lim[3], 12500),
    np.random.uniform(lim[3], lim[4], 6000),
    np.random.uniform(lim[4], lim[5], 3000),
    np.random.uniform(lim[5], lim[6], 1500)
])

# Configuración de parámetros de la simulación
muestras_por_año = {
    2022: 6578,
    2023: 9921,
    2024: 8935,
    2025: 9618
}

# Categorías estándar de Riesgo Operacional (Basilea) y probabilidades teóricas
tipos_evento = [
    'Fraude Externo', 
    'Ejecución, Entrega y Gestión de Procesos',
    'Fallas Tecnológicas e Interrupción del Negocio', 
    'Fraude Interno',
    'Clientes, Productos y Prácticas Empresariales',
    'Relaciones Laborales y Seguridad',
    'Daños a Activos Físicos'
]
prob_tipos = [0.35, 0.25, 0.20, 0.05, 0.10, 0.03, 0.02] 

lineas_negocio = [
    'Banca Minorista (Retail)',
    'Banca Comercial / Corporativa',
    'Tarjetas y Medios de Pago',
    'Tesorería y Mercados',
    'Gestión de Activos'
]
prob_lineas = [0.45, 0.20, 0.25, 0.08, 0.02]

In [ ]:
# Generación dinámica de los DataFrames
dfs = []

for anio, n_muestras in muestras_por_año.items():
    df_temp = pd.DataFrame()
    
    # Identificador único del evento
    df_temp['id_evento'] = [f"EVT-{str(uuid.uuid4())[:8].upper()}" for _ in range(n_muestras)]
    
    # Cuantía de la pérdida (Muestreo sin reemplazo de la población base)
    df_temp['cuantia_perdida'] = np.random.choice(poblacion_cuantia_perdidas, size=n_muestras, replace=False)
    
    # Generación de fechas
    inicio_año = pd.to_datetime(f'{anio}-01-01')
    fin_año = pd.to_datetime(f'{anio}-12-31')
    dias_totales = (fin_año - inicio_año).days
    dias_aleatorios = np.random.randint(0, dias_totales + 1, size=n_muestras)
    df_temp['fecha_evento'] = inicio_año + pd.to_timedelta(dias_aleatorios, unit='D')
    
    # Asignación de variables categóricas con base en probabilidades
    df_temp['tipo_evento'] = np.random.choice(tipos_evento, size=n_muestras, p=prob_tipos)
    df_temp['linea_negocio'] = np.random.choice(lineas_negocio, size=n_muestras, p=prob_lineas)
    
    # Estado del ticket/evento
    df_temp['estado'] = np.random.choice(['Cerrado', 'En Investigación', 'En Recuperación Legal'], 
                                         size=n_muestras, p=[0.75, 0.15, 0.10])
    
    # Simulación de recuperaciones (Solo para eventos cerrados, un % logran recuperar dinero)
    recupera_algo = np.random.choice([0, 1], size=n_muestras, p=[0.6, 0.4]) # 60% no recupera nada
    porcentaje_recuperado = np.random.uniform(0.1, 0.9, size=n_muestras)
    
    df_temp['cuantia_recuperada'] = np.where(
        (df_temp['estado'] == 'Cerrado') & (recupera_algo == 1),
        df_temp['cuantia_perdida'] * porcentaje_recuperado,
        0.0
    )
    
    # Pérdida neta real
    df_temp['perdida_neta'] = df_temp['cuantia_perdida'] - df_temp['cuantia_recuperada']
    
    dfs.append(df_temp)

# 4. Consolidar, ordenar y limpiar
perdoper = pd.concat(dfs, ignore_index=True)
perdoper = perdoper.sort_values('fecha_evento').reset_index(drop=True)

In [ ]:
perdoper.sample(10)

In [ ]:
perdoper['año'] = perdoper['fecha_evento'].dt.year
perdoper['mes'] = perdoper['fecha_evento'].dt.month

# Analisis de las perdidas

In [ ]:
perdoper.head()

In [ ]:
perdoper.cuantia_perdida[perdoper['año']==2023].describe()

In [ ]:
resultado=perdoper.groupby('año')['cuantia_perdida'].agg(['min','mean', 'median',lambda x: x.quantile(0.9),'max','sum','std'])
resultado.rename(columns={'<lambda_0>': 'Cuantil_90'}, inplace=True)
resultado

In [ ]:
# Gráficos distribución observada (empírica)
# ==============================================================================
fig, axs = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))

# Histograma
axs[0].hist(x=perdoper['cuantia_perdida'], color="#3182bd",density=True)
axs[0].set_title('Distribución empírica')
axs[0].set_xlabel('perdidas');axs[0].set_ylabel('density')

# Función de Distribución Acumulada
# ecdf (empirical cumulative distribution function)
ecdf = ECDF(x=perdoper['cuantia_perdida'])
axs[1].plot(ecdf.x, ecdf.y, color="#3182bd")
axs[1].set_title('Función de distribución empírica')
axs[1].set_xlabel('perdidas');axs[1].set_ylabel('CDF')

plt.tight_layout();

In [ ]:
tabla=perdoper.groupby('año')['año'].count()
tabla.plot(kind = 'bar')

# Distribuciones de probabilidad

Variables continuas

In [ ]:
# Extraer los datos limpios
data = perdoper['cuantia_perdida'].dropna()

# Definir la lista de distribuciones a evaluar
distribuciones = [stats.norm, stats.lognorm, stats.gamma, stats.expon]

In [ ]:
# Preparar el lienzo del gráfico
fig, ax = plt.subplots(figsize=(10, 6))

counts, bins, patches = ax.hist(x=data, density=True, bins=50, color="#3182bd", alpha=0.4, label='Datos reales')
limite_y = counts.max() * 1.20

x_hat = np.linspace(min(data), max(data), num=500)
resultados = []

# Iterar sobre cada distribución
for distribucion in distribuciones:
    
    # Ajuste de parámetros
    parametros = distribucion.fit(data)
    
    # Extraer nombres de parámetros
    nombre_parametros = [p for p in inspect.signature(distribucion._pdf).parameters if not p=='x'] + ["loc", "scale"]
    parametros_dict = dict(zip(nombre_parametros, parametros))
    
    # Cálculos de bondad de ajuste
    log_likelihood = distribucion.logpdf(data.to_numpy(), *parametros).sum()
    k = len(parametros)
    n = len(data)
    
    aic = -2 * log_likelihood + 2 * k
    bic = -2 * log_likelihood + np.log(n) * k
    
    # Añadir la curva al gráfico
    y_hat = distribucion.pdf(x_hat, *parametros)
    ax.plot(x_hat, y_hat, linewidth=2.5, label=f'{distribucion.name} (AIC: {aic:.0f})')
    
    # Guardar métricas
    resultados.append({
        'Distribución': distribucion.name,
        'Log-Likelihood': log_likelihood,
        'AIC': aic,
        'BIC': bic,
        'Parámetros': {k: round(v, 4) for k, v in parametros_dict.items()}
    })

ax.set_ylim(0, limite_y)
ax.set_xlim(min(data), max(data))

# Personalizar y mostrar gráfico
ax.set_title('Ajuste de Múltiples Distribuciones a Pérdidas Operativas', fontsize=14, fontweight='bold')
ax.set_xlabel('Cuantía de Pérdida (Millones)')
ax.set_ylabel('Densidad')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar el resumen como un DataFrame ordenado por AIC
df_resultados = pd.DataFrame(resultados).sort_values(by='AIC').reset_index(drop=True)

print("===========================================================")
print(" Resumen de Ajustes (Ordenado por AIC - Menor es Mejor)")
print("===========================================================")
display(df_resultados)

Variables Discretas

In [ ]:
eventos_mensuales = perdoper.groupby(['año', 'mes'])['id_evento'].count()

In [ ]:
print("======================================================")
print(" 1. PRUEBA DE DISTRIBUCIÓN UNIFORME ")
print("======================================================")
# H0: La distribución de eventos es Uniforme (la misma cantidad de eventos cada mes)
f_obs_mes = eventos_mensuales.values
f_exp_mes = [eventos_mensuales.mean()] * len(eventos_mensuales) 

chi2_uni, p_val_uni = chisquare(f_obs=f_obs_mes, f_exp=f_exp_mes)

print(f"Media esperada de eventos por mes: {eventos_mensuales.mean():.2f}")
print(f"Estadístico Chi-Cuadrado: {chi2_uni:.2f}")
print(f"P-valor: {p_val_uni:.4e}")

if p_val_uni < 0.05:
    print("-> RESULTADO: Rechazamos H0. Los eventos mensuales NO siguen una distribución uniforme.")
else:
    print("-> RESULTADO: No podemos rechazar H0. Los eventos mensuales parecen uniformes en el tiempo.")

In [ ]:
print("\n======================================================")
print(" 2. PRUEBA FORMAL DE DISTRIBUCIÓN DE POISSON")
print("======================================================")
# H0: La Varianza es igual a la Media (Sigue una distribución de Poisson)
# H1: La Varianza es distinta a la Media (No sigue Poisson, hay sobre/sub dispersión)

esperanza_mensual = eventos_mensuales.mean()
varianza_mensual = eventos_mensuales.var(ddof=1)
indice_dispersion = varianza_mensual / esperanza_mensual
n_meses = len(eventos_mensuales)

# Prueba de Dispersión de Fisher (Sigue una distribución Chi-Cuadrado)
estadistico_poisson = (n_meses - 1) * indice_dispersion
grados_libertad = n_meses - 1

# Calculamos el P-valor a dos colas
p_valor_poisson = 2 * min(chi2.cdf(estadistico_poisson, grados_libertad), 
                          1 - chi2.cdf(estadistico_poisson, grados_libertad))

print(f"Esperanza Matemática (E[X]): {esperanza_mensual:.2f}")
print(f"Varianza (Var(X)): {varianza_mensual:.2f}")
print(f"Índice de Dispersión: {indice_dispersion:.2f}")
print(f"Estadístico Chi-Cuadrado (Dispersión): {estadistico_poisson:.2f}")
print(f"P-valor: {p_valor_poisson:.4e}")

if p_valor_poisson < 0.05:
    print("-> RESULTADO: Rechazamos H0. La diferencia entre Media y Varianza es estadísticamente significativa. NO es una Poisson.")
else:
    print("-> RESULTADO: No podemos rechazar H0. Hay equidispersión. Los datos SÍ siguen una distribución de Poisson.")

In [ ]:
# --- Visualización Teórica vs Real (Poisson) ---
plt.figure(figsize=(10, 5))
plt.hist(eventos_mensuales, bins=15, density=True, alpha=0.5, color='orange', label='Frecuencia Real (Meses)', edgecolor='black')

x_poisson = np.arange(eventos_mensuales.min(), eventos_mensuales.max())
y_poisson = poisson.pmf(x_poisson, mu=esperanza_mensual) 

plt.plot(x_poisson, y_poisson, 'ro-', ms=5, linewidth=2, label=f'Poisson Teórica (Media={esperanza_mensual:.0f})')
plt.title('Distribución de Frecuencias Mensuales vs Modelo de Poisson', fontsize=13, fontweight='bold')
plt.xlabel('Cantidad de Eventos Operativos en un Mes')
plt.ylabel('Probabilidad P(X=x)')
plt.legend()
plt.tight_layout()
plt.show()

# Convolucion
Supuesto: Eventos identicamente distribuidos

In [ ]:
inicio = time.time()

# Definición de la distribución
mu =  perdoper.groupby('año')['año'].count().mean() # parametro de forma
poisson = stats.poisson(mu)

distribucion = stats.lognorm
parametros   = distribucion.fit(perdoper['cuantia_perdida'].to_numpy())

n=1000

# Convolucion1
perdidas=distribucion.rvs(*parametros, size=n)*poisson.rvs(size=n)

# Guardar el tiempo de finalización
fin = time.time()

# Calcular el tiempo total de ejecución
tiempo_total = fin - inicio

print(f"Tiempo de ejecución: {tiempo_total} segundos")

In [ ]:
sns.kdeplot(x = perdidas, label='Perdidas')

# Calcular y trazar la línea vertical en x=0
plt.axvline(x=perdidas.mean(), color='red', linestyle='--', label="PE= "+str(round(perdidas.mean())))
plt.axvline(x=np.percentile(perdidas,99), color='red', linestyle='-', label="VaR= "+str(round(np.percentile(perdidas,99))))

# Agregar una leyenda
plt.legend()
plt.title("Distribucion de Perdidas Agregadas (Frecuencias Poisson)")
plt.xlabel('Perdidas en Millones')

plt.show()


Supuesto: Eventos no se distribuyen identicamente

In [ ]:
inicio = time.time()

# Definición de la distribución
mu =  perdoper.groupby('año')['año'].count().mean() # parametro de forma
poisson = stats.poisson(mu)

distribucion = stats.lognorm
parametros   = distribucion.fit(perdoper['cuantia_perdida'].to_numpy())

n=1000

perdidas = []

for i in range(n):
    j=distribucion.rvs(*parametros, size=poisson.rvs(size=1)).sum()
    perdidas.append(j)

# Guardar el tiempo de finalización
fin = time.time()

# Calcular el tiempo total de ejecución
tiempo_total = fin - inicio

print(f"Tiempo de ejecución: {tiempo_total} segundos")

In [ ]:
sns.kdeplot(x = perdidas, label='Perdidas')

# Calcular y trazar la línea vertical en x=0
plt.axvline(x=(np.array(perdidas)).mean(), color='red', linestyle='--', label="PE= "+str(round((np.array(perdidas)).mean())))
plt.axvline(x=np.percentile(perdidas,99), color='red', linestyle='-', label="VaR= "+str(round(np.percentile(perdidas,99))))

# Agregar una leyenda
plt.legend()
plt.title("Distribucion de Perdidas Agregadas (Frecuencias Poisson)")
plt.xlabel('Perdidas en Millones')

plt.show()

# Teorema del Limite Central

In [ ]:
# Definición de la distribución
mu =  perdoper.groupby('año')['año'].count().mean() # parametro de forma
poisson = stats.poisson(mu)

distribucion = stats.lognorm
parametros   = distribucion.fit(perdoper['cuantia_perdida'].to_numpy())

# Crear una lista vacía para almacenar los valores
perdesp = []

n=1000 # tamaño de cada escenario
m=100 # tamaño de varios escenarios

for i in range(m):
    j=(distribucion.rvs(*parametros, size=n)*poisson.rvs(size=n)).mean()
    perdesp.append(j)

In [ ]:
sns.kdeplot(x = perdesp, label='PE')

# Una desviacion estandar
plt.axvline(x=(np.array(perdesp)).mean()+(np.array(perdesp)).std(), color='red',
            linestyle='--', label="LimSup "+str(round((np.array(perdesp)).mean()+(np.array(perdesp)).std())))
plt.axvline(x=(np.array(perdesp)).mean()-(np.array(perdesp)).std(), color='red',
            linestyle='-', label="LimInf "+str(round((np.array(perdesp)).mean()-(np.array(perdesp)).std())))
plt.text(303000, 2.5e-5, '68% Datos', fontsize=12, color='black')

# # Dos desviacion estandar
# plt.axvline(x=(np.array(perdesp)).mean()+2*(np.array(perdesp)).std(), color='red',
#             linestyle='--', label="LimSup "+str(round((np.array(perdesp)).mean()+2*(np.array(perdesp)).std())))
# plt.axvline(x=(np.array(perdesp)).mean()-2*(np.array(perdesp)).std(), color='red',
#             linestyle='-', label="LimInf "+str(round((np.array(perdesp)).mean()-2*(np.array(perdesp)).std())))
# plt.text(303000, 2.5e-5, '95% Datos', fontsize=12, color='black')

# # Tres desviacion estandar
# plt.axvline(x=(np.array(perdesp)).mean()+3*(np.array(perdesp)).std(), color='red',
#             linestyle='--', label="LimSup "+str(round((np.array(perdesp)).mean()+3*(np.array(perdesp)).std())))
# plt.axvline(x=(np.array(perdesp)).mean()-3*(np.array(perdesp)).std(), color='red',
#             linestyle='-', label="LimInf "+str(round((np.array(perdesp)).mean()-3*(np.array(perdesp)).std())))
# plt.text(303000, 2.5e-5, '99% Datos', fontsize=12, color='black')

# Agregar una leyenda
plt.legend()
plt.title("Distribucion de Perdidas Agregadas (Frecuencias Uniforme)")
plt.xlabel('Perdidas en Millones')

plt.show()